In [ ]:
import os
from google.colab import drive

# mount point
mount_path = '/content/drive'

if not os.path.exists(mount_path):
    print("Attempting to mount Google Drive...")
    drive.mount(mount_path)
else:
    print("Google Drive is already accessible at:", mount_path)


Attempting to mount Google Drive...
Mounted at /content/drive


In [ ]:
import os

folder_path = "/content/drive/MyDrive/2025-2026 academic year/2026 spring/cs 4485"

if os.path.exists(folder_path):
    print("✅ Path found! Listing files:")
    files = os.listdir(folder_path)
    for f in files:
        print(f"- {f}")
else:
    print("❌ Path not found. Double-check that your Drive is mounted and the spelling is exact.")

✅ Path found! Listing files:
- test_images_labels_targets.tar
- Data pre processing.gdoc


Extraction of the test images folder in google collab

In [ ]:
!tar -xvf "/content/drive/MyDrive/2025-2026 academic year/2026 spring/cs 4485/test_images_labels_targets.tar" -C /content/

Streaming output truncated to the last 5000 lines.
test/labels/midwest-flooding_00000210_pre_disaster.json
test/labels/mexico-earthquake_00000080_pre_disaster.json
test/labels/socal-fire_00000558_pre_disaster.json
test/labels/hurricane-matthew_00000297_pre_disaster.json
test/labels/palu-tsunami_00000139_post_disaster.json
test/labels/hurricane-harvey_00000164_pre_disaster.json
test/labels/socal-fire_00000379_post_disaster.json
test/labels/socal-fire_00001222_post_disaster.json
test/labels/socal-fire_00000882_pre_disaster.json
test/labels/hurricane-matthew_00000076_pre_disaster.json
test/labels/socal-fire_00000823_pre_disaster.json
test/labels/hurricane-florence_00000005_pre_disaster.json
test/labels/socal-fire_00000646_post_disaster.json
test/labels/socal-fire_00001220_post_disaster.json
test/labels/hurricane-harvey_00000328_pre_disaster.json
test/labels/socal-fire_00000749_post_disaster.json
test/labels/hurricane-florence_00000253_pre_disaster.json
test/labels/socal-fire_00001240_pre_

Total Images in each folder for us to check which disaster type has an ideal number of images-Issue#1-pick disaster

In [ ]:
import os


IMAGE_DIR = "/content/test/images"
TARGET_DIR = "/content/test/targets"


DISASTER_PREFIX = "hurricane-florence"

images = [f for f in os.listdir(IMAGE_DIR) if f.startswith(DISASTER_PREFIX)]
targets = [f for f in os.listdir(TARGET_DIR) if f.startswith(DISASTER_PREFIX)]

print(f"--- Audit for {DISASTER_PREFIX} ---")
print(f"Total Images (Pre + Post): {len(images)}")
print(f"Total Targets (Pre + Post): {len(targets)}")


missing_targets = []
for img in images:
    target_name = img.replace(".png", "_target.png") 
    if not os.path.exists(os.path.join(TARGET_DIR, target_name)):
        if not os.path.exists(os.path.join(TARGET_DIR, img)):
            missing_targets.append(img)

print(f"Images missing a target: {len(missing_targets)}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/test/images'

In [ ]:
import os
import json
from collections import defaultdict

SOURCE_LABEL_DIR = "/content/test/labels"
SOURCE_IMAGE_DIR = "/content/test/images"
SOURCE_TARGET_DIR = "/content/test/targets"

print("SOURCE_LABEL_DIR exists:", os.path.exists(SOURCE_LABEL_DIR))
print("Files in SOURCE_LABEL_DIR:", os.listdir(SOURCE_LABEL_DIR))


SOURCE_LABEL_DIR exists: False


FileNotFoundError: [Errno 2] No such file or directory: '/content/test/labels'

Preprocessing step: Created a structure mapping of disaster types and events to their associated label files, pre/post-disaster images, and target images

In [ ]:
def preview_dataset_structure():
    print("🚀 FUNCTION STARTED")

    json_files = [f for f in os.listdir(SOURCE_LABEL_DIR) if f.endswith(".json")]
    print(f"🧾 Found {len(json_files)} JSON files")

    preview_map = defaultdict(lambda: defaultdict(lambda: {
        "labels": [],
        "images": [],
        "targets": []
    }))

    for filename in json_files:
        with open(os.path.join(SOURCE_LABEL_DIR, filename), "r") as f:
            data = json.load(f)

        meta = data.get("metadata", {})

        d_type = meta.get("disaster_type", "flooding").upper()
        d_name = meta.get("disaster", "unknown")

        if "hurricane" in d_name.lower():
            d_type = "HURRICANE"

        post_img = meta.get("img_name", "MISSING.png")
        pre_img = post_img.replace("_post_disaster", "_pre_disaster")
        target_img = post_img.replace(".png", "_target.png")

        preview_map[d_type][d_name]["labels"].append(filename)
        preview_map[d_type][d_name]["images"].extend([pre_img, post_img])
        preview_map[d_type][d_name]["targets"].append(target_img)

    print("\n📊 DATASET PREVIEW STRUCTURE")
    for d_type, disasters in preview_map.items():
        print(f"\n📁 TYPE: {d_type}")
        for d_name, files in disasters.items():
            print(f"   └── 📍 {d_name}")
            print(f"       • Labels:  {len(files['labels'])}")
            print(f"       • Images:  {len(files['images'])}")
            print(f"       • Targets: {len(files['targets'])}")


Created a separate directory structure to organize and test the Hurricane Florence dataset before image cropping was performed.

In [ ]:
import os
import json
import shutil
from collections import defaultdict

#PATHS
SOURCE_LABEL_DIR = "/content/test/labels"
SOURCE_IMAGE_DIR = "/content/test/images"
SOURCE_TARGET_DIR = "/content/test/targets"
DRIVE_ROOT = "/content/drive/MyDrive/CS4485_Project"

# DIRECTORY CREATOR

def create_organized_directories():
    print("📂 Starting Directory Creation & Partitioning...")

    stats = defaultdict(int)
    json_files = [f for f in os.listdir(SOURCE_LABEL_DIR) if f.endswith(".json")]

    for filename in json_files:
        with open(os.path.join(SOURCE_LABEL_DIR, filename), 'r') as f:
            data = json.load(f)
            meta = data['metadata']

            # Determine Disaster Type and Name
            d_type = meta.get('disaster_type', 'flooding').upper()
            d_name = meta.get('disaster', 'unknown')

            # Hurricane logic
            if "hurricane" in d_name.lower():
                d_type = "HURRICANE"

            event_path = os.path.join(DRIVE_ROOT, d_type, d_name)


            subfolders = ['images', 'labels', 'targets']
            for sub in subfolders:
                os.makedirs(os.path.join(event_path, sub), exist_ok=True)

            # 4. Copy Files into the new structure
            shutil.copy(os.path.join(SOURCE_LABEL_DIR, filename), os.path.join(event_path, 'labels', filename))

            # Identify Pre and Post Images
            post_img = meta['img_name']
            pre_img = post_img.replace("_post_disaster", "_pre_disaster")
            target_img = post_img.replace(".png", "_target.png")

            # Copy Images
            for img in [pre_img, post_img]:
                src_img = os.path.join(SOURCE_IMAGE_DIR, img)
                if os.path.exists(src_img):
                    shutil.copy(src_img, os.path.join(event_path, 'images', img))
            src_target = os.path.join(SOURCE_TARGET_DIR, target_img)
            if os.path.exists(src_target):
                shutil.copy(src_target, os.path.join(event_path, 'targets', target_img))

            stats[d_name] += 1

    print("\n--- DIRECTORY CREATION COMPLETE ---")
    for event, count in stats.items():
        print(f"✅ Created structure for: {event} ({count} image sets)")

# Run it
create_organized_directories()

📂 Starting Directory Creation & Partitioning...


KeyboardInterrupt: 

In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/CS4485_Project"

print("--- Final Partition Check ---")
for category in sorted(os.listdir(DRIVE_ROOT)):
    category_path = os.path.join(DRIVE_ROOT, category)
    if os.path.isdir(category_path):
        locations = os.listdir(category_path)
        print(f"📁 {category}: {len(locations)} locations found.")
        for loc in locations:
            img_count = len(os.listdir(os.path.join(category_path, loc, 'images')))
            print(f"   └── 📍 {loc}: {img_count} images synced")

--- Final Partition Check ---
📁 EARTHQUAKE: 1 locations found.
   └── 📍 mexico-earthquake: 46 images synced
📁 FIRE: 2 locations found.
   └── 📍 socal-fire: 415 images synced
   └── 📍 santa-rosa-wildfire: 101 images synced
📁 FLOODING: 1 locations found.
   └── 📍 midwest-flooding: 126 images synced
📁 HURRICANE: 4 locations found.
   └── 📍 hurricane-florence: 153 images synced
   └── 📍 hurricane-harvey: 153 images synced
   └── 📍 hurricane-michael: 138 images synced
   └── 📍 hurricane-matthew: 104 images synced
📁 TSUNAMI: 1 locations found.
   └── 📍 palu-tsunami: 58 images synced
📁 VOLCANO: 1 locations found.
   └── 📍 guatemala-volcano: 7 images synced


In [ ]:
import json
import os

EVENT_PATH = "/content/drive/MyDrive/CS4485_Project/HURRICANE/hurricane-florence"
LABEL_DIR = os.path.join(EVENT_PATH, "labels")
TARGET_POST_IMAGE = "hurricane-florence_00000007_post_disaster.png"

label_file = TARGET_POST_IMAGE.replace(".png", ".json")
label_path = os.path.join(LABEL_DIR, label_file)

with open(label_path, "r") as f:
    data = json.load(f)

# See what keys exist
print("Top-level keys in JSON:", list(data.keys()))

print(data["features"].keys())



Top-level keys in JSON: ['features', 'metadata']
dict_keys(['lng_lat', 'xy'])


Performed an initial cropping test on a single Hurricane Florence image using building polygons stored in WKT format from the original JSON structure

In [ ]:
import os
from PIL import Image
from shapely import wkt
import json

# CONFIG
EVENT_PATH = "/content/drive/MyDrive/CS4485_Project/HURRICANE/hurricane-florence"
TARGET_POST_IMAGE = "hurricane-florence_00000007_post_disaster.png"
BUFFER = 45  # medium context

# PATHS
IMAGE_DIR = os.path.join(EVENT_PATH, "images")
LABEL_DIR = os.path.join(EVENT_PATH, "labels")
TARGET_DIR = os.path.join(EVENT_PATH, "targets")

CROP_ROOT = os.path.join(
    EVENT_PATH,
    "crops",
    "medium",
    TARGET_POST_IMAGE.replace(".png", "")
)
os.makedirs(CROP_ROOT, exist_ok=True)

# LOAD JSON
label_file = TARGET_POST_IMAGE.replace(".png", ".json")
label_path = os.path.join(LABEL_DIR, label_file)

with open(label_path, "r") as f:
    data = json.load(f)

# EXTRACT PIXEL POLYGONS
polygons = []
for feature in data["features"]["xy"]:
    if feature["properties"]["feature_type"] == "building":
        poly_wkt = feature["wkt"]
        poly = wkt.loads(poly_wkt)
        coords = list(poly.exterior.coords)
        polygons.append(coords)

# LOAD IMAGES
pre_name = TARGET_POST_IMAGE.replace("_post_disaster", "_pre_disaster")
target_name = TARGET_POST_IMAGE.replace(".png", "_target.png")

pre_img = Image.open(os.path.join(IMAGE_DIR, pre_name)).convert("RGB")
post_img = Image.open(os.path.join(IMAGE_DIR, TARGET_POST_IMAGE)).convert("RGB")
target_img = Image.open(os.path.join(TARGET_DIR, target_name)).convert("RGB")

# CROP EACH HOUSE
for i, poly in enumerate(polygons, start=1):
    xs = [p[0] for p in poly]
    ys = [p[1] for p in poly]

    x1 = max(0, min(xs) - BUFFER)
    y1 = max(0, min(ys) - BUFFER)
    x2 = min(pre_img.width, max(xs) + BUFFER)
    y2 = min(pre_img.height, max(ys) + BUFFER)

    pre_crop = pre_img.crop((x1, y1, x2, y2))
    post_crop = post_img.crop((x1, y1, x2, y2))
    target_crop = target_img.crop((x1, y1, x2, y2))

    house_dir = os.path.join(CROP_ROOT, f"house_{i:03d}")
    os.makedirs(house_dir, exist_ok=True)

    pre_crop.save(os.path.join(house_dir, "pre.png"))
    post_crop.save(os.path.join(house_dir, "post.png"))
    target_crop.save(os.path.join(house_dir, "target.png"))

print("✅ Done!")
print(f"📁 Crops saved to:\n{CROP_ROOT}")


KeyboardInterrupt: 

In [ ]:
import os
import json
from PIL import Image
from shapely import wkt
from shapely.geometry import Polygon


BUFFER = 45
EVENT_PATH = "/content/drive/MyDrive/CS4485_Project/HURRICANE/hurricane-florence"

# External Output Path (To keep source clean)
OUTPUT_BASE = "/content/drive/MyDrive/CS4485_Organized_Crops"

# Source Folders
IMAGE_DIR = os.path.join(EVENT_PATH, "images")
LABEL_DIR = os.path.join(EVENT_PATH, "labels")
TARGET_DIR = os.path.join(EVENT_PATH, "targets")

os.makedirs(OUTPUT_BASE, exist_ok=True)


# Get all post-disaster images in the directory
post_images = [f for f in os.listdir(IMAGE_DIR) if "_post_disaster.png" in f]

print(f"🚀 Found {len(post_images)} post-disaster images. Starting batch process...")

for post_filename in post_images:
    image_id = post_filename.replace(".png", "")

    # Define matching filenames
    pre_filename = post_filename.replace("_post_disaster", "_pre_disaster")
    target_filename = post_filename.replace(".png", "_target.png")
    json_filename = post_filename.replace(".png", ".json")

    # Set up output paths for this specific image
    image_output_root = os.path.join(OUTPUT_BASE, image_id)
    crop_folder = os.path.join(image_output_root, "crops")
    os.makedirs(crop_folder, exist_ok=True)

    # Path checking
    if not all([os.path.exists(os.path.join(IMAGE_DIR, pre_filename)),
                os.path.exists(os.path.join(TARGET_DIR, target_filename)),
                os.path.exists(os.path.join(LABEL_DIR, json_filename))]):
        print(f"⚠️ Skipping {post_filename}: Missing matching pre, target, or json files.")
        continue

    pre_img = Image.open(os.path.join(IMAGE_DIR, pre_filename)).convert("RGB")
    post_img = Image.open(os.path.join(IMAGE_DIR, post_filename)).convert("RGB")
    target_img = Image.open(os.path.join(TARGET_DIR, target_filename)).convert("RGB")

    with open(os.path.join(LABEL_DIR, json_filename), "r") as f:
        original_data = json.load(f)

    # Master JSON structure for THIS image
    master_json_output = {
        "metadata": original_data.get("metadata", {}),
        "houses": {}
    }

    # ==========================================
    # 4. CROP HOUSES
    # ==========================================
    house_count = 0
    # Access the building features
    for feature in original_data["features"]["xy"]:
        if feature["properties"]["feature_type"] == "building":
            house_count += 1
            house_key = f"house_{house_count:03d}"

            # Extract geometry
            poly = wkt.loads(feature["wkt"])
            xs, ys = poly.exterior.xy

            # Calculate crop bounds
            x1 = max(0, int(min(xs) - BUFFER))
            y1 = max(0, int(min(ys) - BUFFER))
            x2 = min(post_img.width, int(max(xs) + BUFFER))
            y2 = min(post_img.height, int(max(ys) + BUFFER))

            # Create subfolder for this house
            house_subfolder = os.path.join(crop_folder, house_key)
            os.makedirs(house_subfolder, exist_ok=True)

            # Save the Triplets
            pre_img.crop((x1, y1, x2, y2)).save(os.path.join(house_subfolder, "pre.png"))
            post_img.crop((x1, y1, x2, y2)).save(os.path.join(house_subfolder, "post.png"))
            target_img.crop((x1, y1, x2, y2)).save(os.path.join(house_subfolder, "target.png"))

            # Shift coordinates for the local crop
            shifted_coords = [(p[0] - x1, p[1] - y1) for p in list(poly.exterior.coords)]
            shifted_poly = Polygon(shifted_coords)

            # Store in master dictionary
            master_json_output["houses"][house_key] = {
                "local_wkt": shifted_poly.wkt,
                "original_wkt": feature["wkt"],
                "properties": feature["properties"]
            }

    # Save the consolidated label file for this specific image
    with open(os.path.join(image_output_root, "labels.json"), "w") as f:
        json.dump(master_json_output, f, indent=4)

    print(f"✅ Finished {image_id}: {house_count} houses cropped.")

print(f"\n✨ ALL DONE! Dataset created at: {OUTPUT_BASE}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/CS4485_Project/HURRICANE/hurricane-florence/images'

Generated fixed-size 256×256 building-centered crops from pre- and post-disaster images and updated polygon annotations to local crop coordinates. #issue 8

In [ ]:
import os
import json
from PIL import Image
from shapely import wkt
from shapely.geometry import Polygon


CROP_SIZE = 256  # This ensures every "Medium" crop is exactly 256x256
HALF_SIZE = CROP_SIZE // 2
EVENT_PATH = "/content/drive/MyDrive/CS4485_Project/HURRICANE/hurricane-florence"
OUTPUT_BASE = "/content/drive/MyDrive/CS4485_Organized_Crops_2"

IMAGE_DIR = os.path.join(EVENT_PATH, "images")
LABEL_DIR = os.path.join(EVENT_PATH, "labels")
TARGET_DIR = os.path.join(EVENT_PATH, "targets")

os.makedirs(OUTPUT_BASE, exist_ok=True)


post_images = [f for f in os.listdir(IMAGE_DIR) if "_post_disaster.png" in f]

for post_filename in post_images:
    image_id = post_filename.replace(".png", "")
    pre_filename = post_filename.replace("_post_disaster", "_pre_disaster")
    target_filename = post_filename.replace(".png", "_target.png")
    json_filename = post_filename.replace(".png", ".json")

    image_output_root = os.path.join(OUTPUT_BASE, image_id)
    crop_folder = os.path.join(image_output_root, "crops")
    os.makedirs(crop_folder, exist_ok=True)

    # Check for file existence
    if not all(os.path.exists(os.path.join(d, f)) for d, f in zip([IMAGE_DIR, TARGET_DIR, LABEL_DIR], [pre_filename, target_filename, json_filename])):
        continue

    # Load Images
    pre_img = Image.open(os.path.join(IMAGE_DIR, pre_filename)).convert("RGB")
    post_img = Image.open(os.path.join(IMAGE_DIR, post_filename)).convert("RGB")
    target_img = Image.open(os.path.join(TARGET_DIR, target_filename)).convert("RGB")
    W, H = post_img.size

    with open(os.path.join(LABEL_DIR, json_filename), "r") as f:
        original_data = json.load(f)

    master_json_output = {"metadata": original_data.get("metadata", {}), "houses": {}}

 
    house_count = 0
    for feature in original_data["features"]["xy"]:
        if feature["properties"]["feature_type"] == "building":
            house_count += 1
            house_key = f"house_{house_count:03d}"

            poly = wkt.loads(feature["wkt"])
            centroid = poly.centroid # Get the center point of the house
            cx, cy = int(centroid.x), int(centroid.y)

            # Define 256x256 box around center
            x1, y1 = cx - HALF_SIZE, cy - HALF_SIZE
            x2, y2 = cx + HALF_SIZE, cy + HALF_SIZE

            # Edge Handling (if house is near the border of the 1024x1024 image)
            if x1 < 0: x1, x2 = 0, CROP_SIZE
            if y1 < 0: y1, y2 = 0, CROP_SIZE
            if x2 > W: x1, x2 = W - CROP_SIZE, W
            if y2 > H: y1, y2 = H - CROP_SIZE, H

            # Save Crops
            house_subfolder = os.path.join(crop_folder, house_key)
            os.makedirs(house_subfolder, exist_ok=True)

            pre_img.crop((x1, y1, x2, y2)).save(os.path.join(house_subfolder, "pre.png"))
            post_img.crop((x1, y1, x2, y2)).save(os.path.join(house_subfolder, "post.png"))
            target_img.crop((x1, y1, x2, y2)).save(os.path.join(house_subfolder, "target.png"))

            # Update Labels with local coordinates
            shifted_coords = [(p[0] - x1, p[1] - y1) for p in list(poly.exterior.coords)]
            master_json_output["houses"][house_key] = {
                "local_wkt": Polygon(shifted_coords).wkt,
                "properties": feature["properties"]
            }

    with open(os.path.join(image_output_root, "labels.json"), "w") as f:
        json.dump(master_json_output, f, indent=4)

    print(f"✅ Processed {image_id}: {house_count} fixed-size crops.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/CS4485_Project/HURRICANE/hurricane-florence/images'

Tested a grid-based tiling approach because building crop sizes varied across images. #issue 8

In [ ]:
import os
import json
from PIL import Image

# ==========================================
# 1. CONFIGURATION
# ==========================================
TILE_SIZE = 256  # The "Medium" size we discussed
STRIDE = 256     # Set to 256 for no overlap, or 128 for 50% overlap
EVENT_PATH = "/content/drive/MyDrive/CS4485_Project/HURRICANE/hurricane-florence"


OUTPUT_BASE = "/content/drive/MyDrive/CS4485_Full_Grid_Crops"

IMAGE_DIR = os.path.join(EVENT_PATH, "images")
TARGET_DIR = os.path.join(EVENT_PATH, "targets")

os.makedirs(OUTPUT_BASE, exist_ok=True)


post_images = [f for f in os.listdir(IMAGE_DIR) if "_post_disaster.png" in f]

print(f"🚀 Starting Full Grid Tiling for {len(post_images)} images...")

for post_filename in post_images:
    image_id = post_filename.replace(".png", "")
    pre_filename = post_filename.replace("_post_disaster", "_pre_disaster")
    target_filename = post_filename.replace(".png", "_target.png")

    image_output_root = os.path.join(OUTPUT_BASE, image_id)
    os.makedirs(image_output_root, exist_ok=True)

    # Load Images
    try:
        pre_img = Image.open(os.path.join(IMAGE_DIR, pre_filename)).convert("RGB")
        post_img = Image.open(os.path.join(IMAGE_DIR, post_filename)).convert("RGB")
        target_img = Image.open(os.path.join(TARGET_DIR, target_filename)).convert("RGB")
    except FileNotFoundError:
        print(f"⚠️ Missing pre or target for {post_filename}, skipping.")
        continue

    W, H = post_img.size
    tile_count = 0


    # Loop through the image width and height in steps of STRIDE
    for y in range(0, H - TILE_SIZE + 1, STRIDE):
        for x in range(0, W - TILE_SIZE + 1, STRIDE):
            tile_count += 1
            tile_name = f"tile_{tile_count:03d}_x{x}_y{y}"

            # Define the folder for this specific grid square
            tile_folder = os.path.join(image_output_root, tile_name)
            os.makedirs(tile_folder, exist_ok=True)

            # Crop and Save the Triplet
            crop_box = (x, y, x + TILE_SIZE, y + TILE_SIZE)

            pre_img.crop(crop_box).save(os.path.join(tile_folder, "pre.png"))
            post_img.crop(crop_box).save(os.path.join(tile_folder, "post.png"))
            target_img.crop(crop_box).save(os.path.join(tile_folder, "target.png"))

    print(f"✅ Generated {tile_count} grid tiles for {image_id}")

print(f"\n✨ DONE! Full grid dataset created at: {OUTPUT_BASE}")

🚀 Starting Full Grid Tiling for 63 images...
✅ Generated 16 grid tiles for hurricane-florence_00000046_post_disaster
✅ Generated 16 grid tiles for hurricane-florence_00000135_post_disaster
✅ Generated 16 grid tiles for hurricane-florence_00000177_post_disaster
✅ Generated 16 grid tiles for hurricane-florence_00000301_post_disaster
✅ Generated 16 grid tiles for hurricane-florence_00000167_post_disaster
✅ Generated 16 grid tiles for hurricane-florence_00000364_post_disaster
✅ Generated 16 grid tiles for hurricane-florence_00000200_post_disaster
✅ Generated 16 grid tiles for hurricane-florence_00000384_post_disaster
✅ Generated 16 grid tiles for hurricane-florence_00000363_post_disaster
✅ Generated 16 grid tiles for hurricane-florence_00000507_post_disaster
✅ Generated 16 grid tiles for hurricane-florence_00000029_post_disaster
✅ Generated 16 grid tiles for hurricane-florence_00000019_post_disaster
✅ Generated 16 grid tiles for hurricane-florence_00000187_post_disaster
✅ Generated 16 grid

Extracted building-level crops and generated updated JSON annotations for each house #issue 40

In [ ]:
import os
import json
from PIL import Image
from shapely import wkt
from shapely.geometry import Polygon


BUFFER = 45
EVENT_PATH = "/content/drive/MyDrive/CS4485_Project/HURRICANE/hurricane-florence"
OUTPUT_BASE = "/content/drive/MyDrive/CS4485_Hurricane_Org_final"

# Source Folders
IMAGE_DIR = os.path.join(EVENT_PATH, "images")
LABEL_DIR = os.path.join(EVENT_PATH, "labels")
TARGET_DIR = os.path.join(EVENT_PATH, "targets")

os.makedirs(OUTPUT_BASE, exist_ok=True)


# Filter for post-disaster images to use as the base for matching
post_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith("_post_disaster.png")]

print(f"🚀 Found {len(post_images)} images. Starting extraction...")

for post_filename in post_images:

    image_id = post_filename.replace(".png", "")
    pre_filename = post_filename.replace("_post_disaster", "_pre_disaster")
    target_filename = post_filename.replace(".png", "_target.png")
    json_filename = post_filename.replace(".png", ".json")


    paths_to_check = [
        os.path.join(IMAGE_DIR, pre_filename),
        os.path.join(TARGET_DIR, target_filename),
        os.path.join(LABEL_DIR, json_filename)
    ]

    if not all(os.path.exists(p) for p in paths_to_check):
        print(f"⚠️ Skipping {image_id}: Missing pre-image, target, or JSON label.")
        continue


    image_output_root = os.path.join(OUTPUT_BASE, image_id)
    os.makedirs(image_output_root, exist_ok=True)


    pre_img = Image.open(os.path.join(IMAGE_DIR, pre_filename)).convert("RGB")
    post_img = Image.open(os.path.join(IMAGE_DIR, post_filename)).convert("RGB")
    target_img = Image.open(os.path.join(TARGET_DIR, target_filename)).convert("RGB")

    with open(os.path.join(LABEL_DIR, json_filename), "r") as f:
        original_data = json.load(f)

    # Master JSON for this specific source image
    master_json_output = {
        "source_image": post_filename,
        "metadata": original_data.get("metadata", {}),
        "houses": {}
    }


    house_count = 0
    for feature in original_data["features"]["xy"]:
        if feature["properties"]["feature_type"] == "building":
            house_count += 1
            house_key = f"house_{house_count:03d}"

            # Get geometry
            poly = wkt.loads(feature["wkt"])
            xs, ys = poly.exterior.xy

            # Calculate crop bounds (with buffer)
            x1 = max(0, int(min(xs) - BUFFER))
            y1 = max(0, int(min(ys) - BUFFER))
            x2 = min(post_img.width, int(max(xs) + BUFFER))
            y2 = min(post_img.height, int(max(ys) + BUFFER))


            house_subfolder = os.path.join(image_output_root, house_key)
            os.makedirs(house_subfolder, exist_ok=True)

            crop_box = (x1, y1, x2, y2)
            pre_img.crop(crop_box).save(os.path.join(house_subfolder, "pre.png"))
            post_img.crop(crop_box).save(os.path.join(house_subfolder, "post.png"))
            target_img.crop(crop_box).save(os.path.join(house_subfolder, "target.png"))


            shifted_coords = [(p[0] - x1, p[1] - y1) for p in list(poly.exterior.coords)]
            shifted_poly = Polygon(shifted_coords)

            # Store house details in the master list
            master_json_output["houses"][house_key] = {
                "local_wkt": shifted_poly.wkt,
                "original_wkt": feature["wkt"],
                "damage_level": feature["properties"].get("subtype", "unknown"),
                "crop_bounds": {"x1": x1, "y1": y1, "x2": x2, "y2": y2}
            }

    with open(os.path.join(image_output_root, "labels.json"), "w") as f:
        json.dump(master_json_output, f, indent=4)

    print(f"✅ Processed {image_id}: {house_count} houses extracted.")

print(f"\n✨ ALL DONE! Dataset ready at: {OUTPUT_BASE}")

🚀 Found 63 images. Starting extraction...
✅ Processed hurricane-florence_00000046_post_disaster: 155 houses extracted.
✅ Processed hurricane-florence_00000135_post_disaster: 5 houses extracted.
✅ Processed hurricane-florence_00000177_post_disaster: 0 houses extracted.
✅ Processed hurricane-florence_00000301_post_disaster: 17 houses extracted.
✅ Processed hurricane-florence_00000167_post_disaster: 4 houses extracted.
✅ Processed hurricane-florence_00000364_post_disaster: 79 houses extracted.
✅ Processed hurricane-florence_00000200_post_disaster: 21 houses extracted.
✅ Processed hurricane-florence_00000384_post_disaster: 42 houses extracted.
✅ Processed hurricane-florence_00000363_post_disaster: 5 houses extracted.
✅ Processed hurricane-florence_00000507_post_disaster: 8 houses extracted.
✅ Processed hurricane-florence_00000029_post_disaster: 5 houses extracted.
✅ Processed hurricane-florence_00000019_post_disaster: 159 houses extracted.
✅ Processed hurricane-florence_00000187_post_disas

Applied the same JSON annotation generation process used for house crops to the tile-based dataset.

In [ ]:
import os
import json
import time
from PIL import Image
from shapely import wkt
from shapely.geometry import Polygon, box, MultiPolygon

TILE_SIZE = 256
STRIDE = 256  # No overlap
EVENT_PATH = "/content/drive/MyDrive/CS4485_Project/HURRICANE/hurricane-florence"
OUTPUT_BASE = "/content/drive/MyDrive/CS4485_Hurricane_Grid_Final"

# Source Folders
IMAGE_DIR = os.path.join(EVENT_PATH, "images")
LABEL_DIR = os.path.join(EVENT_PATH, "labels")
TARGET_DIR = os.path.join(EVENT_PATH, "targets")

os.makedirs(OUTPUT_BASE, exist_ok=True)

post_images = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith("_post_disaster.png")])

print(f"🚀 Found {len(post_images)} images. Starting Grid Tiling + Master Labeling...")

for post_filename in post_images:
    image_id = post_filename.replace(".png", "")
    pre_filename = post_filename.replace("_post_disaster", "_pre_disaster")
    target_filename = post_filename.replace(".png", "_target.png")
    json_filename = post_filename.replace(".png", ".json")

    image_output_root = os.path.join(OUTPUT_BASE, image_id)
    os.makedirs(image_output_root, exist_ok=True)

    # Load Images & Metadata
    try:
        pre_img = Image.open(os.path.join(IMAGE_DIR, pre_filename)).convert("RGB")
        post_img = Image.open(os.path.join(IMAGE_DIR, post_filename)).convert("RGB")
        target_img = Image.open(os.path.join(TARGET_DIR, target_filename)).convert("RGB")

        with open(os.path.join(LABEL_DIR, json_filename), "r") as f:
            original_data = json.load(f)
    except FileNotFoundError:
        print(f"⚠️ Skipping {image_id}: Missing files.")
        continue

    # Master structure for this image
    master_grid_output = {
        "source_image": post_filename,
        "metadata": original_data.get("metadata", {}),
        "tiles": {}
    }

    # Pre-load all buildings from the original JSON
    all_buildings = []
    for feature in original_data["features"]["xy"]:
        if feature["properties"]["feature_type"] == "building":
            poly = wkt.loads(feature["wkt"])
            if poly.is_valid:
                all_buildings.append({"poly": poly, "properties": feature["properties"]})

    W, H = post_img.size
    tile_count = 0

    # 3. GRID SLICING
    for y in range(0, H - TILE_SIZE + 1, STRIDE):
        for x in range(0, W - TILE_SIZE + 1, STRIDE):
            tile_count += 1
            tile_id = f"tile_{tile_count:03d}"

            x1, y1, x2, y2 = x, y, x + TILE_SIZE, y + TILE_SIZE
            tile_box = box(x1, y1, x2, y2)

            # Initialize tile entry in master JSON
            master_grid_output["tiles"][tile_id] = {
                "crop_bounds": {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
                "buildings": []
            }

            # Spatial Check: Which buildings are in this tile?
            for b in all_buildings:
                if tile_box.intersects(b["poly"]):
                    intersection = tile_box.intersection(b["poly"])
                    if not intersection.is_empty:
                        # Handle Polygon/MultiPolygon
                        if isinstance(intersection, Polygon):
                            polys = [intersection]
                        elif isinstance(intersection, MultiPolygon):
                            polys = list(intersection.geoms)
                        else:
                            continue

                        for p in polys:
                            # Shift coordinates relative to the tile top-left (x1, y1)
                            shifted_coords = [(pt[0] - x1, pt[1] - y1) for pt in list(p.exterior.coords)]
                            master_grid_output["tiles"][tile_id]["buildings"].append({
                                "local_wkt": Polygon(shifted_coords).wkt,
                                "damage_level": b["properties"].get("subtype", "unknown")
                            })

            # Create tile folder and save images
            tile_folder = os.path.join(image_output_root, tile_id)
            os.makedirs(tile_folder, exist_ok=True)

            crop_box = (x1, y1, x2, y2)
            try:
                pre_img.crop(crop_box).save(os.path.join(tile_folder, "pre.png"))
                post_img.crop(crop_box).save(os.path.join(tile_folder, "post.png"))
                target_img.crop(crop_box).save(os.path.join(tile_folder, "target.png"))
            except Exception as e:
                print(f"❌ Error saving images for {tile_id}: {e}")

    # Save the consolidated label file for this image
    with open(os.path.join(image_output_root, "master_grid_labels.json"), "w") as f:
        json.dump(master_grid_output, f, indent=4)

    print(f"✅ Processed {image_id}: {tile_count} tiles generated.")

print(f"\n✨ ALL DONE! Dataset ready at: {OUTPUT_BASE}")

🚀 Found 63 images. Starting Grid Tiling + Master Labeling...
✅ Processed hurricane-florence_00000004_post_disaster: 16 tiles generated.
✅ Processed hurricane-florence_00000007_post_disaster: 16 tiles generated.
✅ Processed hurricane-florence_00000019_post_disaster: 16 tiles generated.
✅ Processed hurricane-florence_00000029_post_disaster: 16 tiles generated.
✅ Processed hurricane-florence_00000039_post_disaster: 16 tiles generated.
✅ Processed hurricane-florence_00000046_post_disaster: 16 tiles generated.
✅ Processed hurricane-florence_00000095_post_disaster: 16 tiles generated.
✅ Processed hurricane-florence_00000112_post_disaster: 16 tiles generated.
✅ Processed hurricane-florence_00000113_post_disaster: 16 tiles generated.
✅ Processed hurricane-florence_00000125_post_disaster: 16 tiles generated.
✅ Processed hurricane-florence_00000128_post_disaster: 16 tiles generated.
✅ Processed hurricane-florence_00000131_post_disaster: 16 tiles generated.
✅ Processed hurricane-florence_00000135

Applied the new JSON annotation structure and parsed cropped building images to generate structured metadata for each house.

In [ ]:
import os
import json
import uuid
from PIL import Image
from shapely import wkt
from shapely.geometry import Polygon

BUFFER      = 45
EVENT_PATH  = "/content/drive/MyDrive/CS4485_Project/HURRICANE/hurricane-florence"
OUTPUT_BASE = "/content/drive/MyDrive/CS4485_Hurricane_Cursor_House"

# Source Folders
IMAGE_DIR  = os.path.join(EVENT_PATH, "images")
LABEL_DIR  = os.path.join(EVENT_PATH, "labels")
TARGET_DIR = os.path.join(EVENT_PATH, "targets")

os.makedirs(OUTPUT_BASE, exist_ok=True)

post_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith("_post_disaster.png")]
print(f"🚀 Found {len(post_images)} images. Starting extraction...")

for post_filename in post_images:

    image_id      = post_filename.replace(".png", "")
    pre_filename  = post_filename.replace("_post_disaster", "_pre_disaster")
    target_filename = post_filename.replace(".png", "_target.png")
    json_filename = post_filename.replace(".png", ".json")

    # Extract pair ID from filename 
    pair_id = image_id.split("_")[2] if len(image_id.split("_")) > 2 else image_id

    # Verification
    paths_to_check = [
        os.path.join(IMAGE_DIR, pre_filename),
        os.path.join(TARGET_DIR, target_filename),
        os.path.join(LABEL_DIR, json_filename)
    ]
    if not all(os.path.exists(p) for p in paths_to_check):
        print(f"⚠️  Skipping {image_id}: Missing pre-image, target, or JSON label.")
        continue

    # 2b. Setup Output Structure
    image_output_root = os.path.join(OUTPUT_BASE, image_id)
    os.makedirs(image_output_root, exist_ok=True)

    # 2c. Load Images & Label Data
    pre_img    = Image.open(os.path.join(IMAGE_DIR, pre_filename)).convert("RGB")
    post_img   = Image.open(os.path.join(IMAGE_DIR, post_filename)).convert("RGB")
    target_img = Image.open(os.path.join(TARGET_DIR, target_filename)).convert("RGB")

    with open(os.path.join(LABEL_DIR, json_filename), "r") as f:
        original_data = json.load(f)

    
    meta         = original_data.get("metadata", {})
    disaster_id  = meta.get("disaster", "hurricane-florence")


    master_json_output = {
        "disasterId": disaster_id,
        "pairId":     pair_id,
        "uid":        f"{disaster_id}-{pair_id}",
        "metadata":   meta,
        "path": {
            "pre":    f"images/{disaster_id}/{pair_id}_pre.png",
            "post":   f"images/{disaster_id}/{pair_id}_post.png",
            "target": f"images/{disaster_id}/{pair_id}_target.png",
        },
        "houses": {}
    }


    # HOUSE-BY-HOUSE CROPPING
   
    house_count = 0

    # Try to get lat/long mapping if available in the original JSON
    # Some xBD datasets include a "metadata" with geo transform info
    geo_features = original_data.get("features", {})
    lng_features = geo_features.get("lng_lat", [])  # lat/long version of features if present

    # Build a lookup from wkt → lat/long points if lng_lat features exist
    wkt_to_latlng = {}
    for feat in lng_features:
        if feat["properties"]["feature_type"] == "building":
            wkt_to_latlng[feat["properties"].get("uid", "")] = feat.get("wkt", "")

    for feature in original_data["features"]["xy"]:
        if feature["properties"]["feature_type"] == "building":
            house_count += 1
            house_key    = f"house_{house_count:03d}"
            building_uid = str(uuid.uuid4())  # unique ID per building

            # Get pixel geometry
            poly    = wkt.loads(feature["wkt"])
            xs, ys  = poly.exterior.xy

            # Calculate crop bounds with buffer
            x1 = max(0, int(min(xs) - BUFFER))
            y1 = max(0, int(min(ys) - BUFFER))
            x2 = min(post_img.width,  int(max(xs) + BUFFER))
            y2 = min(post_img.height, int(max(ys) + BUFFER))

            # Create house subfolder
            house_subfolder = os.path.join(image_output_root, house_key)
            os.makedirs(house_subfolder, exist_ok=True)

            # Save pre, post, target crops
            crop_box = (x1, y1, x2, y2)
            pre_img.crop(crop_box).save(os.path.join(house_subfolder,    "pre.png"))
            post_img.crop(crop_box).save(os.path.join(house_subfolder,   "post.png"))
            target_img.crop(crop_box).save(os.path.join(house_subfolder, "target.png"))

            # Shift coordinates so (0,0) = top-left of crop
            shifted_coords = [(p[0] - x1, p[1] - y1) for p in list(poly.exterior.coords)]

            lat_lng_poly   = None
            lng_lat_feats  = original_data.get("features", {}).get("lng_lat", [])
            if house_count - 1 < len(lng_lat_feats):
                try:
                    lat_lng_poly = wkt.loads(lng_lat_feats[house_count - 1]["wkt"])
                except Exception:
                    lat_lng_poly = None

            def build_points(coords, lat_lng_poly=None):
                """Build list of {x, y, lat, long} for each polygon vertex."""
                points = []
                ll_coords = list(lat_lng_poly.exterior.coords) if lat_lng_poly else []
                for i, (px, py) in enumerate(coords):
                    point = {"x": round(px, 10), "y": round(py, 10)}
                    if ll_coords and i < len(ll_coords):
                        point["lat"]  = round(ll_coords[i][1], 15)  
                        point["long"] = round(ll_coords[i][0], 15)  
                    else:
                        point["lat"]  = None
                        point["long"] = None
                    points.append(point)
                return points

            # Original (full image) pixel coords
            original_coords = list(poly.exterior.coords)

            # Store in new format
            master_json_output["houses"][house_key] = {
                "uid":            building_uid,
                "type":           "building",
                "classification": "unknown",      
                "prediction":     None,            
                "crop_bounds": {
                    "x1": x1, "y1": y1,
                    "x2": x2, "y2": y2
                },
                "points": {
                    # Shifted coords (relative to crop) 
                    "local": build_points(shifted_coords, lat_lng_poly),
                    "original": build_points(original_coords, lat_lng_poly),
                }
            }

    # Save the new format labels.json
    with open(os.path.join(image_output_root, "labels.json"), "w") as f:
        json.dump(master_json_output, f, indent=4)

    print(f"  ✅ {image_id}: {house_count} houses extracted.")

print(f"\n✨ ALL DONE! Dataset ready at: {OUTPUT_BASE}")

🚀 Found 63 images. Starting extraction...
  ✅ hurricane-florence_00000046_post_disaster: 155 houses extracted.
  ✅ hurricane-florence_00000135_post_disaster: 5 houses extracted.
  ✅ hurricane-florence_00000177_post_disaster: 0 houses extracted.
  ✅ hurricane-florence_00000301_post_disaster: 17 houses extracted.
  ✅ hurricane-florence_00000167_post_disaster: 4 houses extracted.
  ✅ hurricane-florence_00000364_post_disaster: 79 houses extracted.
  ✅ hurricane-florence_00000200_post_disaster: 21 houses extracted.
  ✅ hurricane-florence_00000384_post_disaster: 42 houses extracted.
  ✅ hurricane-florence_00000363_post_disaster: 5 houses extracted.
  ✅ hurricane-florence_00000507_post_disaster: 8 houses extracted.
  ✅ hurricane-florence_00000029_post_disaster: 5 houses extracted.
  ✅ hurricane-florence_00000019_post_disaster: 159 houses extracted.
  ✅ hurricane-florence_00000187_post_disaster: 55 houses extracted.
  ✅ hurricane-florence_00000128_post_disaster: 1 houses extracted.
  ✅ hurrica

Did the samething but for tiles

In [ ]:
import os
import json
import uuid
from PIL import Image
from shapely import wkt
from shapely.geometry import Polygon, box, MultiPolygon


TILE_SIZE  = 256
STRIDE     = 256  
EVENT_PATH  = "/content/drive/MyDrive/CS4485_Project/HURRICANE/hurricane-florence"
OUTPUT_BASE = "/content/drive/MyDrive/CS4485_Hurricane_Crop_Tiles"


IMAGE_DIR  = os.path.join(EVENT_PATH, "images")
LABEL_DIR  = os.path.join(EVENT_PATH, "labels")
TARGET_DIR = os.path.join(EVENT_PATH, "targets")

os.makedirs(OUTPUT_BASE, exist_ok=True)


post_images = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith("_post_disaster.png")])
print(f"🚀 Found {len(post_images)} images. Starting Grid Tiling...")

for post_filename in post_images:
    image_id        = post_filename.replace(".png", "")
    pre_filename    = post_filename.replace("_post_disaster", "_pre_disaster")
    target_filename = post_filename.replace(".png", "_target.png")
    json_filename   = post_filename.replace(".png", ".json")

    # Extract pair ID 
    parts   = image_id.split("_")
    pair_id = parts[2] if len(parts) > 2 else image_id

    image_output_root = os.path.join(OUTPUT_BASE, image_id)
    os.makedirs(image_output_root, exist_ok=True)

    # Load Images & Metadata
    try:
        pre_img    = Image.open(os.path.join(IMAGE_DIR, pre_filename)).convert("RGB")
        post_img   = Image.open(os.path.join(IMAGE_DIR, post_filename)).convert("RGB")
        target_img = Image.open(os.path.join(TARGET_DIR, target_filename)).convert("RGB")

        with open(os.path.join(LABEL_DIR, json_filename), "r") as f:
            original_data = json.load(f)
    except FileNotFoundError:
        print(f"⚠️  Skipping {image_id}: Missing files.")
        continue


    meta        = original_data.get("metadata", {})
    disaster_id = meta.get("disaster", "hurricane-florence")

    # ── New format master JSON
    master_grid_output = {
        "disasterId": disaster_id,
        "pairId":     pair_id,
        "uid":        f"{disaster_id}-{pair_id}",
        "metadata":   meta,
        "path": {
            "pre":    f"images/{disaster_id}/{pair_id}_pre.png",
            "post":   f"images/{disaster_id}/{pair_id}_post.png",
            "target": f"images/{disaster_id}/{pair_id}_target.png",
        },
        "tiles": {}
    }

    # ── Pre-load all buildings from original JSON 
    # Also grab lng_lat features for lat/long coords if available
    all_buildings  = []
    lng_lat_feats  = original_data.get("features", {}).get("lng_lat", [])

    for i, feature in enumerate(original_data["features"]["xy"]):
        if feature["properties"]["feature_type"] == "building":
            poly = wkt.loads(feature["wkt"])
            if poly.is_valid:
                # Try to get matching lat/long polygon
                lat_lng_poly = None
                if i < len(lng_lat_feats):
                    try:
                        lat_lng_poly = wkt.loads(lng_lat_feats[i]["wkt"])
                    except Exception:
                        lat_lng_poly = None

                all_buildings.append({
                    "poly":        poly,
                    "lat_lng_poly": lat_lng_poly,
                    "properties":  feature["properties"]
                })

    W, H       = post_img.size
    tile_count = 0


    # GRID SLICING (unchanged logic)

    for y in range(0, H - TILE_SIZE + 1, STRIDE):
        for x in range(0, W - TILE_SIZE + 1, STRIDE):
            tile_count += 1
            tile_id = f"tile_{tile_count:03d}"

            x1, y1, x2, y2 = x, y, x + TILE_SIZE, y + TILE_SIZE
            tile_box = box(x1, y1, x2, y2)

            # ── Initialize tile in new format 
            master_grid_output["tiles"][tile_id] = {
                "uid":        str(uuid.uuid4()),  # unique tile ID
                "crop_bounds": {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
                "path": {
                    "pre":    f"images/{disaster_id}/{pair_id}/tiles/{tile_id}/pre.png",
                    "post":   f"images/{disaster_id}/{pair_id}/tiles/{tile_id}/post.png",
                    "target": f"images/{disaster_id}/{pair_id}/tiles/{tile_id}/target.png",
                },
                "buildings": []
            }

            for b in all_buildings:
                if tile_box.intersects(b["poly"]):
                    intersection = tile_box.intersection(b["poly"])
                    if not intersection.is_empty:

                        # Handle Polygon or MultiPolygon
                        if isinstance(intersection, Polygon):
                            polys = [intersection]
                        elif isinstance(intersection, MultiPolygon):
                            polys = list(intersection.geoms)
                        else:
                            continue

                        for p in polys:
                            # Shift coords relative to tile top-left (x1, y1)
                            shifted_coords   = [(pt[0] - x1, pt[1] - y1) for pt in list(p.exterior.coords)]
                            original_coords  = list(p.exterior.coords)

                            # Try to get lat/long coords for this building
                            lat_lng_poly = b.get("lat_lng_poly")
                            ll_coords    = list(lat_lng_poly.exterior.coords) if lat_lng_poly else []

                            # ── Build points arrays 
                            def build_points(coords, ll_coords=None):
                                points = []
                                for i, (px, py) in enumerate(coords):
                                    point = {
                                        "x": round(px, 10),
                                        "y": round(py, 10),
                                        "lat":  round(ll_coords[i][1], 15) if ll_coords and i < len(ll_coords) else None,
                                        "long": round(ll_coords[i][0], 15) if ll_coords and i < len(ll_coords) else None,
                                    }
                                    points.append(point)
                                return points

                            master_grid_output["tiles"][tile_id]["buildings"].append({
                                "uid":            str(uuid.uuid4()),  # unique building ID
                                "type":           "building",
                                "classification": "unknown",          # VLM fills this in
                                "prediction":     None,               # VLM fills this in
                                "points": {
                                    # Local = relative to tile crop (what model sees)
                                    "local":    build_points(shifted_coords,  ll_coords),
                                    # Original = relative to full 1024x1024 image
                                    "original": build_points(original_coords, ll_coords),
                                }
                            })

            #  Save tile images (unchanged) 
            tile_folder = os.path.join(image_output_root, tile_id)
            os.makedirs(tile_folder, exist_ok=True)

            crop_box = (x1, y1, x2, y2)
            try:
                pre_img.crop(crop_box).save(os.path.join(tile_folder,    "pre.png"))
                post_img.crop(crop_box).save(os.path.join(tile_folder,   "post.png"))
                target_img.crop(crop_box).save(os.path.join(tile_folder, "target.png"))
            except Exception as e:
                print(f"❌ Error saving images for {tile_id}: {e}")

    with open(os.path.join(image_output_root, "master_grid_labels.json"), "w") as f:
        json.dump(master_grid_output, f, indent=4)

    print(f"  ✅ {image_id}: {tile_count} tiles generated.")

print(f"\n✨ ALL DONE! Dataset ready at: {OUTPUT_BASE}")

🚀 Found 63 images. Starting Grid Tiling...
  ✅ hurricane-florence_00000004_post_disaster: 16 tiles generated.
  ✅ hurricane-florence_00000007_post_disaster: 16 tiles generated.
  ✅ hurricane-florence_00000019_post_disaster: 16 tiles generated.
  ✅ hurricane-florence_00000029_post_disaster: 16 tiles generated.
  ✅ hurricane-florence_00000039_post_disaster: 16 tiles generated.
  ✅ hurricane-florence_00000046_post_disaster: 16 tiles generated.
  ✅ hurricane-florence_00000095_post_disaster: 16 tiles generated.
  ✅ hurricane-florence_00000112_post_disaster: 16 tiles generated.
  ✅ hurricane-florence_00000113_post_disaster: 16 tiles generated.
  ✅ hurricane-florence_00000125_post_disaster: 16 tiles generated.
  ✅ hurricane-florence_00000128_post_disaster: 16 tiles generated.
  ✅ hurricane-florence_00000131_post_disaster: 16 tiles generated.
  ✅ hurricane-florence_00000135_post_disaster: 16 tiles generated.
  ✅ hurricane-florence_00000141_post_disaster: 16 tiles generated.
  ✅ hurricane-flore